Generate Time-Dependent Covariates

# Setup

In [ ]:
library(tidyverse)
library(bigrquery)
library(tictoc)
library(data.table)
library(survival)
source("functions.R")

In [ ]:
tic("Time to run script")

In [ ]:
med_concepts_key <- c( 
    "BPDrug" = "21600381, 21601461, 21601560, 21601664, 21601744, 21601782", #bp_lowering_drugs
    "Insulin" = "21600713", #"insulin"
    "Metformin" = "1503297", #"metformin"
    "Glucagon" = "1560278", #"glucagon"
    "DMDrug" = "1123618, 1123627, 21600749, 21600765, 21600775, 21600779, 21600783, 21600788", #oraldm (may also be glp1 here)
    "LipidDrug"= "21601853", #lipidsmod
    "Anticoagulants" = "1310149, 1315865, 21600972, 21601019, 43534760",
    "Aspirin" = "1112807",
    "Corticosteroids" = "21600652, 21602723, 21602745, 21603283",
    "ProtonPumpInhibitors" = "21600095",
    "Antipsychotics" = "21604490",
    "OpioidForPain_Rx" = "1102527, 1103314, 1110410, 1124957, 1126658, 1201620, 21604255, 21604269, 21604275, 21604283, 21604286, 21604288"
)
med_concepts_key_df <- data.frame(feature_name = names(med_concepts_key), Concept_Ancestor_IDs = med_concepts_key)

med_patterns_key <- c(
    "ACEInhibitors" = "pril",
    "AngiotensinIIRB" = "artan",
    "CChannelBlockers" = "pine|zem|plendil|verapamil",
    "BetaBlockers" = "olol|alol|ilol",
    "Diuretics" = "furosemide|indapamide|acetazolamide|metolazone|thiazide|triamterene|amiloride|chlorthalidone|torsemide|diuril|spironolactone|eplerenone|dyrenium|edecrin|bumetanide|ethacryn",
    #other_bplowering is anything not above
    "Sulfonylurea" = "piride|pizide|glyburide|tolazamide|tolbutamide|chlorpropamide",
    "DPP4" = "gliptin",
    "SGLT2" = "flozin",
    "Thiazoladinedione" = "glitazone",
    "GLP1" = "glutide|exenatide|lixisenatide|semaglutide|tirzepatide",
    #otherglucoselowering is anything not above
    "Statin_Any" = "statin",
    "Niacin" = "niacin",
    "Ezetimibe" = "ezetimibe", 
    "Fibrates" = "gemfibrozil|fibrate|fibric", 
    "BAS" = "colestipol|cholestyramine|colesevelam",
    "FattyAcid" = "docosahexaeno|omega-fattyacids"
    #otherlipidslowering is anything not above
)

dose_patterns_key <- c(
       "Statin_High" = "simvastatin 80|atorvastatin [48]0|rosuvastatin [24]0|rosuvastatin calcium [24]0", 
       "Statin_Medium"  = "simvastatin [24]0|atorvastatin [12]0|rosuvastatin (5|10)|rosuvastatin calcium (5|10)|pravastatin [48]0|pravastatin sodium [48]0|lovastatin [46]0|fluvastatin 80|pitavastatin [1-4]|pitavastatin calcium [1-4]|pitavastatin magnesium [1-4]", 
       "Statin_Low" = "simvastatin (5|10)|atorvastatin 5|rosuvastatin 2\\.5|rosuvastatin calcium 2\\.5|lovastatin [12]0|pravastatin (5|10|20)|pravastatin sodium (5|10|20)|fluvastatin [24]0"
    )


med_type <- 
     data.frame(feature_parent = "BPDrug", 
                feature_name = c("ACEInhibitors", "AngiotensinIIRB", "CChannelBlockers", "BetaBlockers", "Diuretics")) %>%
    bind_rows(
        data.frame(feature_parent = "DMDrug",
                   feature_name = c("Sulfonylurea", "DPP4", "SGLT2", "Thiazoladinedione", "GLP1" )) 
        ) %>%
    bind_rows(
        data.frame(feature_parent = "LipidDrug",
                   feature_name = c("Statin_Any", "Niacin", "Ezetimibe", "Fibrates", "BAS", "Statin_High", 
                                    "Statin_Medium", "Statin_Low"))    
    )

In [ ]:
med_master_key <- med_concepts_key_df %>%
    full_join(med_type %>% dplyr::rename(feature_name2 = feature_name) %>%
              mutate(feature_name = feature_parent)) %>%
    mutate(feature_name = coalesce(feature_name2, feature_name)) %>%
    select(-feature_name2) %>%
    mutate(pattern = med_patterns_key[feature_name],
          dose_pattern = dose_patterns_key[feature_name],
           field_name = "standard_concept_name")

In [ ]:
med_master_key %>% mutate(pattern = paste0(substr(pattern, 0, 10), "...."))

In [ ]:
write_to_bucket(med_master_key, "medications_concept_ancestors_patterns_AOU.csv")

In [ ]:
#get via med_concepts_key["med_of_interest"]
length(med_concepts_key)

In [ ]:
med_concepts_key

In [ ]:
med_patterns_key

In [ ]:
dose_patterns_key

# Conditions

## Define conditions key 

In [ ]:
conditions_patterns_ICD9 <- 
    data.frame(feature_name = "Backpain", patterns = c("'720.%'", "'721.%'", "'722.%'", "'723.%'", "'724.%'")) %>%
    bind_rows(data.frame(feature_name = "EH_COMDIAB", patterns =  c("'250.4%'", "'250.5%'", "'250.6%'", "'250.7%'", "'250.9%'"))) %>%
    bind_rows(data.frame(feature_name = "EH_RENAL", patterns = c("'403.11%'", "'403.91%'",
                                              "'404.12%'", "'404.92%'",
                                              "'585.%'","'586.%'",
                                              "'v42.0%'","'v45.1%'",
                                              "'v56.4%'", "'v56.8%'"))) %>%
    bind_rows(data.frame(feature_name = "EH_ARRHYTH", patterns = c("'426.2%'", "'426.3%'","'426.4%'","'426.6%'","'426.7%'","'426.8%'",
                                               "'426.10%'",  "'426.11%'",  "'426.13%'", 
                                               "'427.0%'", "'427.2%'","'427.9%'",
                                               "'427.31%'", 
                                               "'427.60%'", "'785.0%'", "'v45.0%'", "'v53.3%'", 
                                               "'426.50%'", "'426.51%'","'426.52%'","'426.53%'"))) %>% 
    bind_rows(data.frame(feature_name = "EH_HYPERTENS", patterns = c("'401.1%'", "'401.9%'", 
                                                      "'402.10%'", "'402.90%'",
                                                      "'404.10%'", "'404.90%'",
                                                      "'405.1%'", "'405.9%'"))) %>%
    bind_rows(data.frame(feature_name = "ODEPRdx_poss", patterns = c("'301.12%'", "'300.4%'", "'293.83%'", "'298.0%'", "'301.1%'",
                                                      "'311.%'", "'296.9%'", "'309.0%'", "'309.1%'", "'296.82%'"))) %>%
    bind_rows(data.frame(feature_name = "EH_CHRNPULM", patterns = c("'490.%'", "'491.%'","'492.%'","'493.%'","'494.%'","'495.%'","'496.%'",
                                                     "'506.4%'","'500.%'","'501.%'","'502.%'", 
                                                     "'503.%'","'504.%'","'505.%'"))) %>%
    bind_rows(data.frame(feature_name = "EH_LIVER", patterns = c("'070.32%'","'070.33%'",
                                                  "'070.54%'", 
                                                  "'456.0%'", "'456.1%'","'456.2%'",
                                                  "'571.0%'", "'571.2%'", "'571.3%'", "'571.5%'", 
                                                              "'571.6%'", "'571.8%'", "'571.9%'", 
                                                  "'571.40%'",  "'571.49%'", 
                                                  "'572.3%'", "'572.8%'",
                                                  "'v42.7%'"))) %>%
    bind_rows(data.frame(feature_name = "EH_ELECTRLYTE", patterns = c("'276%'"))) %>%
    bind_rows(data.frame(feature_name = "EH_OBESITY", patterns = c("'278.0%'"))) %>%
    bind_rows(data.frame(feature_name = "Proteinuria_Dx", patterns = c("'593.6%'","'791.0%'"))) %>%
    bind_rows(data.frame(feature_name = "SleepDisorder", patterns = c("'307.4%'","'327%'",  "'347%'",  "'780.5%'"))) %>%
    bind_rows(data.frame(feature_name = "Neuropathy", patterns = c("'337.0'","'337.00%'",  "'337.09%'",  "'337.1%'", 
                                                                  "'356%'", "'357.2%'",  "'250.61%'",  "'250.63%'",
                                                                   "'250.60%'", "'250.62%'" ))) %>%
    mutate(field_name = "source_concept_code_icd9")

conditions_patterns_ICD10 <- 
     data.frame(feature_name = "Backpain", patterns = c("'G54.1%'", "'G54.3%'","'G54.4%'",
                         "'M43.0%'", "'M43.1%'", "'M43.3%'", "'M43.5%'", "'M43.8%'", "'M43.9%'", 
                         "'M45.0%'", "'M45.4%'", "'M45.5%'", "'M45.6%'", "'M45.7%'", "'M45.8%'", "'M45.9%'", 
                         "'M46.0%'", "'M46.1%'","'M46.4%'","'M46.8%'","'M46.9%'", 
                          "'M47%'", "'M48%'","'M49%'",
                         "'M51%'",  "'M53%'",  "'M54%'", 
                         "'Q76%'", "'S23%'", "'S33%'")) %>%
    bind_rows(data.frame(feature_name = "EH_COMDIAB", patterns = c("'E10.0%'", "'E10.1%'","'E10.2%'","'E10.3%'","'E10.4%'","'E10.5%'",
                                                 "'E11.0%'", "'E11.1%'","'E11.2%'","'E11.3%'","'E11.4%'","'E11.5%'"))) %>%
    bind_rows(data.frame(feature_name = "EH_RENAL", patterns = c("'I12.0%'", "'I13.1%'",
                                                  "'N18.1%'","'N18.2%'","'N18.3%'","'N18.4%'","'N18.5%'","'N18.6%'","'N18.9%'",
                                                  "'N19%'", "'N25.0%'","'Z49.0%'","'Z94.0%'","'Z99.2%'"))) %>%
    bind_rows(data.frame(feature_name = "EH_ARRHYTH", patterns = c("'I48%'",
                                                 "'I49%'"))) %>% 
    bind_rows(data.frame(feature_name = "EH_HYPERTENS", patterns = c("'I10%'", "'I11%'"))) %>%
    bind_rows(data.frame(feature_name = "ODEPRdx_poss", patterns = c("'F32%'", "'F33%'"))) %>%
    bind_rows(data.frame(feature_name = "EH_CHRNPULM", patterns = c("'J40%'", "'J41%'","'J42%'","'J43%'","'J44%'","'J6%'"))) %>%
    bind_rows(data.frame(feature_name = "EH_LIVER", patterns = c("'K70%'", "'K71%'", "'K72%'", "'K73%'", "'K74%'", "'K75%'", "'K76%'", 
                                                  "'B18.%'", 
                                                  "'I85%'"))) %>%
    bind_rows(data.frame(feature_name = "EH_ELECTRLYTE", patterns = c("'E22.2%'",                                               
                                                  "'E86.0%'", 
                                                  "'E86.1%'", 
                                                  "'E86.9%'",   
                                                  "'E87.0%'",
                                                  "'E87.1%'",
                                                  "'E87.2%'",
                                                  "'E87.3%'",
                                                  "'E87.4%'",
                                                  "'E87.5%'",
                                                  "'E87.6%'",
                                                  "'E87.7%'",
                                                  "'E87.8%'"
                                                 ))) %>%
    bind_rows(data.frame(feature_name = "EH_OBESITY", patterns = c("'E66%'"))) %>%
    bind_rows(data.frame(feature_name = "Proteinuria_Dx", patterns = c("'R80.0%'",
                                                  "'R80.1%'",
                                                  "'R80.2%'",
                                                  "'R80.3%'",
                                                  "'R80.8%'",
                                                  "'R80.9%'",
                                                  "'N06%'"))) %>%
    bind_rows(data.frame(feature_name = "SleepDisorder", patterns = c("'G47%'"))) %>%
    bind_rows(data.frame(feature_name = "Neuropathy", patterns = c("'G99.0%'",  "'G90.0'",
                                                                   "'G90.09%'", "'G60%'",  "'E10.42%'",  "'E08.42%'",
                                                                   "'E09.42%'", "'E10.4%'",  "'E11.4%'"))) %>%
    mutate(field_name = "source_concept_code_icd10")

conditions_patterns_key_df <- bind_rows(conditions_patterns_ICD9, 
                                       conditions_patterns_ICD10) %>%
    arrange(feature_name)

getICD9pattern <- function(x) { 
    conditions_patterns_key_df %>% filter(feature_name == x & field_name == "source_concept_code_icd9") %>%
        .$patterns
}
getICD10pattern  <- function(x) { 
    conditions_patterns_key_df %>% filter(feature_name == x & field_name == "source_concept_code_icd10") %>%
        .$patterns
}

In [ ]:
conditions_patterns_key_df

In [ ]:
conditions_patterns_key_out <- conditions_patterns_key_df %>%
    group_by(feature_name, field_name) %>%
    summarise(patterns = paste0(patterns, collapse = ", "))
conditions_patterns_key_out

In [ ]:
write_to_bucket(conditions_patterns_key_out,  "conditions_sqlpatterns_AOU.csv")

## Pull conditions

### Neuropathy

In [ ]:
Neuropathy <- extract_condition(
    where_ICD9_like = getICD9pattern("Neuropathy"), 
    where_ICD10_like = getICD10pattern("Neuropathy"), 
    name = "Neuropathy") 

In [ ]:
Neuropathy$all %>% count(Neuropathy_concept_source_code, Neuropathy_concept_name, sort=T) 

### Back Pain

In [ ]:
Backpain <- extract_condition(
    where_ICD9_like = getICD9pattern("Backpain"), 
    where_ICD10_like = getICD10pattern("Backpain"), 
    name = "Backpain")

In [ ]:
Backpain$first %>% count(Backpain_concept_source_code, Backpain_concept_name, sort=T) 

### Complicated Diabetes

In [ ]:
EH_COMDIAB <- extract_condition(
    where_ICD9_like = getICD9pattern("EH_COMDIAB"), 
    where_ICD10_like = getICD10pattern("EH_COMDIAB"), 
    name = "EH_COMDIAB")

In [ ]:
EH_COMDIAB$first %>% count(EH_COMDIAB_concept_source_code, EH_COMDIAB_concept_name, sort=T)

### Renal Conditions

In [ ]:
# EH_RENAL <- extract_condition(
#     where_ICD9_like = getICD9pattern("EH_RENAL"), 
#     where_ICD10_like = getICD10pattern("EH_RENAL"), 
#     name = "EH_RENAL")

In [ ]:
# EH_RENAL$first %>% count(EH_RENAL_concept_source_code, EH_RENAL_concept_name, sort=T) 

### Arrhythmias

In [ ]:
EH_ARRHYTH <- extract_condition(
    where_ICD9_like = getICD9pattern("EH_ARRHYTH"), 
    where_ICD10_like = getICD10pattern("EH_ARRHYTH"), 
    name = "EH_ARRHYTH")

In [ ]:
EH_ARRHYTH$first %>% count(EH_ARRHYTH_concept_source_code, EH_ARRHYTH_concept_name, sort=T) 

### Hypertension

In [ ]:
EH_HYPERTENS <- extract_condition(
    where_ICD9_like = getICD9pattern("EH_HYPERTENS"), 
    where_ICD10_like = getICD10pattern("EH_HYPERTENS"), 
    name = "EH_HYPERTENS")

In [ ]:
EH_HYPERTENS$first %>% count(EH_HYPERTENS_concept_source_code, EH_HYPERTENS_concept_name, sort=T)

### Possible Depression

In [ ]:
ODEPRdx_poss <- extract_condition(
    where_ICD9_like = getICD9pattern("ODEPRdx_poss"), 
    where_ICD10_like = getICD10pattern("ODEPRdx_poss"), 
    name = "ODEPRdx_poss")

In [ ]:
ODEPRdx_poss$first %>% count(ODEPRdx_poss_concept_source_code, ODEPRdx_poss_concept_name, sort=T) 

### Chronic Pulmonary conditions

In [ ]:
EH_CHRNPULM <- extract_condition(
    where_ICD9_like = getICD9pattern("EH_CHRNPULM"), 
    where_ICD10_like = getICD10pattern("EH_CHRNPULM"), 
    name = "EH_CHRNPULM")

In [ ]:
EH_CHRNPULM$first %>% count(EH_CHRNPULM_concept_source_code, EH_CHRNPULM_concept_name, sort= T)

### Liver Conditions

In [ ]:
EH_LIVER <- extract_condition(
    where_ICD9_like = getICD9pattern("EH_LIVER"), 
    where_ICD10_like = getICD10pattern("EH_LIVER"), 
    name = "EH_LIVER")

In [ ]:
EH_LIVER$first %>% count(EH_LIVER_concept_source_code, EH_LIVER_concept_name, sort= T)

### Diagnoses of Electrolyte Imbalances

In [ ]:
EH_ELECTRLYTE <- extract_condition(
    where_ICD9_like = getICD9pattern("EH_ELECTRLYTE"), 
    where_ICD10_like = getICD10pattern("EH_ELECTRLYTE"), 
    name = "EH_ELECTRLYTE")

In [ ]:
EH_ELECTRLYTE$first %>% count(EH_ELECTRLYTE_concept_source_code, EH_ELECTRLYTE_concept_name, sort= T)

### Diagnoses of Obesity

In [ ]:
EH_OBESITY <- extract_condition(
    where_ICD9_like = getICD9pattern("EH_OBESITY"), 
    where_ICD10_like = getICD10pattern("EH_OBESITY"), 
    name = "EH_OBESITY")

In [ ]:
EH_OBESITY$first %>% count(EH_OBESITY_concept_source_code, EH_OBESITY_concept_name, sort= T)

### Diagnoses of Proteinuria

In [ ]:
Proteinuria_Dx <- extract_condition(
    where_ICD9_like = getICD9pattern("Proteinuria_Dx"), 
    where_ICD10_like = getICD10pattern("Proteinuria_Dx"), 
    name = "Proteinuria_Dx")

In [ ]:
Proteinuria_Dx$first %>% count(Proteinuria_Dx_concept_source_code, Proteinuria_Dx_concept_name, sort= T)

### Diagnoses of Sleep Disorders

In [ ]:
SleepDisorder <- extract_condition(
    where_ICD9_like = getICD9pattern("SleepDisorder"), 
    where_ICD10_like = getICD10pattern("SleepDisorder"), 
    name = "SleepDisorder")

In [ ]:
SleepDisorder$first %>% count(SleepDisorder_concept_source_code, SleepDisorder_concept_name, sort= T)

# Procedures

## Define procedure key

In [ ]:
procedure_codes_df <- 
    data.frame(feature_name = "doppler", code = c("'93307'", "'93320'", "'93325'", "'C8929'")) %>%
    mutate(field_name = "source_concept_code_procedure")

getProcedureCode <- function(x) { 
    procedure_codes_df %>% filter(feature_name == x) %>%
        .$code
}

In [ ]:
procedure_codes_df_out <- procedure_codes_df %>%
    group_by(feature_name) %>% 
    summarise(code = paste0(code, collapse = ", "))
procedure_codes_df_out

In [ ]:
write_to_bucket(procedure_codes_df_out, "procedures_conceptcodes_AOU.csv")

## Pull Procedures

### Doppler

In [ ]:
doppler <- extract_procedure(code_in = c("'93307'", "'93320'", "'93325'", "'C8929'"), name="doppler")

In [ ]:
doppler$first %>% count(doppler_concept_source_code, doppler_concept_name, sort=T)

# Save time-dependent conditions and procedures object

In [ ]:
object_names <- c("doppler", "Backpain", "EH_COMDIAB", "EH_ARRHYTH", 
                  #"EH_RENAL", 
                  "EH_HYPERTENS", "ODEPRdx_poss", "EH_CHRNPULM", "EH_LIVER", 
                 "EH_ELECTRLYTE", "EH_OBESITY", "Proteinuria_Dx", "SleepDisorder")

write_RDATA_to_bucket(object_names, destination_filename = "timedep_conditions_and_procedures.RData")

In [ ]:
rm(list = object_names)
gc()

# DCSI Score

## Pull subcomponents

### Retinopathy subcomponent

In [ ]:
dcsi.ret1 <- extract_condition(
    where_ICD9_like = c(
        "'250.5%'", 
        "'249.5%'",                                                  
        "'362.00%'", "'362.01%'", "'362.03%'", "'362.04%'", "'362.05%'", "'362.06%'", "'362.07%'", "'362.08%'", "'362.09%'",                                                    
        "'362.1%'", 
        "'362.53%'",                                 
        "'362.81%'", "'362.82%'", "'362.83%'" ), 
    where_ICD10_like = c(
        "'E08.30%'",  "'E08.31%'", "'E08.32%'", "'E08.33%'", "'E08.36%'", "'E08.37%'", "'E08.38%'", "'E08.39%'",                                                                              
        "'E09.30%'", "'E09.31%'", "'E09.32%'", "'E09.33%'", "'E09.36%'", "'E09.37%'", "'E09.38%'", "'E09.39%'", 
        "'E10.30%'", "'E10.31%'", "'E10.32%'", "'E10.33%'", "'E10.36%'", "'E10.37%'", "'E10.38%'", "'E10.39%'", 
        "'E11.30%'","'E11.31%'","'E11.32%'","'E11.33%'","'E11.36%'","'E11.37%'","'E11.38%'","'E11.39%'",
        "'E13.30%'","'E13.31%'","'E13.32%'","'E13.33%'","'E13.36%'","'E13.37%'","'E13.38%'","'E13.39%'",
        "'H35.0%'", 
        "'H35.6%'", 
        "'H35.9%'", 
        "'H35.35%'", 
        "'H35.8%'"), 
    name = "dcsi.ret1") 

dcsi.ret2 <- extract_condition(where_ICD9_like = c("'361%'",
                                                 "'369%'",
                                                 "'362.02%'",
                                                 "'379.23%'"), 
                             where_ICD10_like =  c("'E08.34%'", "'E08.35%'",
                                                   "'E09.34%'", "'E09.35%'",
                                                   "'E10.34%'", "'E10.35%'",
                                                   "'E11.34%'", "'E11.35%'",
                                                   "'E13.34%'", "'E13.35%'" ), 
                             name = "dcsi.ret2")

In [ ]:
dcsi.ret1$first %>% count(dcsi.ret1_concept_source_code, dcsi.ret1_concept_name, sort=T)

In [ ]:
dcsi.ret2$first %>% count(dcsi.ret2_concept_source_code, dcsi.ret2_concept_name, sort=T) 

### Nephropathy subcomponent

In [ ]:
dcsi.neph1 <- extract_condition(
    where_ICD9_like = c(
        "'580%'", "'581%'", "'582%'", "'583%'",
        "'585.1%'", "'585.2%'", "'585.3%'", "'585.9%'",
        "'250.4%'",
        "'249.4%'"), 
    where_ICD10_like = c(
        "'E08.21%'", "'E08.22%'", "'E08.29%'",                                                                              
        "'E09.21%'", "'E09.22%'", "'E09.29%'", 
        "'E10.21%'", "'E10.22%'", "'E10.29%'", 
        "'E11.21%'", "'E11.22%'", "'E11.29%'",
        "'E13.21%'", "'E13.22%'", "'E13.29%'",
        "'N00%'", "'N03%'", "'N04%'", "'N05%'",
        "'N18.1%'", "'N18.2%'", "'N18.3%'", "'N18.9%'"), 
     name = "dcsi.neph1")

dcsi.neph2 <- extract_condition(
    where_ICD9_like = c("'585.4%'", "'585.5%'", "'585.6%'",
                       "'586%'",
                       "'593.9%'"), 
    where_ICD10_like = c("'N18.4%'", "'N18.5%'", "'N18.6%'",
                         "'N19%'"), 
    name = "dcsi.neph2")

In [ ]:
dcsi.neph1$first %>% count(dcsi.neph1_concept_source_code, dcsi.neph1_concept_name, sort=T)

In [ ]:
dcsi.neph2$first %>% count(dcsi.neph2_concept_source_code, dcsi.neph2_concept_name, sort=T)

### Neuropathy subcomponent

In [ ]:
dcsi.neur1 <- extract_condition(
    where_ICD9_like = c(
        "'249.6%'",
        "'250.6%'",
        "'337.0%'", "'337.1%'",
        "'354%'", "'355%'",
        "'356.9%'",
        "'357.2%'",
        "'358.1%'",
        "'458.0%'",
        "'536.3%'",
        "'564.5%'",
        "'596.54%'",
        "'713.5%'",
        "'951.0%'",
        "'951.1%'",
        "'951.3%'" ), 
    where_ICD10_like = c("'E08.4%'", "'E09.4%'", "'E10.4%'", "'E11.4%'", "'E13.4%'", 
         "'G56%'", "'G57%'",
         "'G60.9%'",
         "'G73.3%'",
         "'G90.01%'", "'G90.09%'",
         "'G90.8%'", "'G90.9%'",
         "'G99.0%'",
         "'H49%'",
         "'I95.1%'",
         "'K31.84%'",
         "'K59.1%'",
         "'N31.9%'",
         "'M14.6%'",
         "'S04%'" ), 
    name = "dcsi.neur1")

In [ ]:
dcsi.neur1$first %>% count(dcsi.neur1_concept_source_code, dcsi.neur1_concept_name, sort=T)

Orthostatic hypotension is indeed included here 

### Cerebrovascular subcomponent

In [ ]:
dcsi.cbr1 <- extract_condition(where_ICD9_like = c("'435%'"), 
                             where_ICD10_like = c("'G45%'"), 
                             name = "dcsi.cbr1")
dcsi.cbr2 <- extract_condition(where_ICD9_like = c("'431%'", "'433%'", "'434%'", "'436%'"), 
                             where_ICD10_like = c("'I61%'", "'I63%'", "'I65%'", "'I66%'", 
                                                 "'I67.81%'"), 
                             name = "dcsi.cbr2")

In [ ]:
dcsi.cbr1$first %>% count(dcsi.cbr1_concept_source_code, dcsi.cbr1_concept_name, sort=T)

In [ ]:
dcsi.cbr2$first %>% count(dcsi.cbr2_concept_source_code, dcsi.cbr2_concept_name, sort=T)

### CVD subcomponent

In [ ]:
dcsi.cvd1 <- extract_condition(
    where_ICD9_like = c(
        "'411%'", "'413%'","'414%'",
        "'429.3%'",
        "'440.0%'","'440.1%'","'440.3%'","'440.4%'","'440.5%'","'440.6%'","'440.7%'","'440.8%'","'440.9%'",
        "'440.20%'","'440.21%'","'440.22%'","'440.25%'","'440.26%'","'440.27%'","'440.28%'","'440.29%'"), 
    where_ICD10_like = c(
        "'I20%'", 
        "'I24%'", 
        "'I25.0%'", "'I25.1%'", "'I25.3%'", "'I25.4%'", "'I25.5%'", "'I25.6%'", "'I25.7%'", "'I25.8%'", "'I25.9%'",
        "'I70.0%'", "'I70.1%'", "'I70.3%'", "'I70.4%'", "'I70.5%'", "'I70.6%'", "'I70.7%'", "'I70.8%'", "'I70.9%'", 
        "'I70.20%'", "'I70.21%'", "'I70.22%'", "'I70.23%'", "'I70.24%'", "'I70.27%'", "'I70.28%'", "'I70.29%'"), 
    name = "dcsi.cvd1")


dcsi.cvd2 <- extract_condition(
    where_ICD9_like = c("'410%'",  "'412%'",                       
                       "'427.1%'", "'427.3%'", "'427.4%'", "'427.5%'",                     
                       "'428%'",                  
                       "'440.23%'",  "'%440.24'",                   
                       "'441%'" ), 
    where_ICD10_like = c("'I21%'", "'I22%'", "'I23%'",
                          "'I25.2%'", "'I46%'", "'I47%'", "'I48%'", "'I49%'",
                          "'I50%'",
                          "'I70.25%'", "'I70.26%'",
                          "'I71%'"), 
    name = "dcsi.cvd2")

In [ ]:
dcsi.cvd1$first %>% count(dcsi.cvd1_concept_source_code, dcsi.cvd1_concept_name, sort=T)

In [ ]:
dcsi.cvd2$first %>% count(dcsi.cvd2_concept_source_code, dcsi.cvd2_concept_name, sort=T)

### Peripheral Vascular Disease subcomponent

In [ ]:
dcsi.pvd1 <- extract_condition(
    where_ICD9_like = c(
      "'250.7%'", 
      "'249.7%'", 
      "'440.21%'", 
      "'442.3%'", 
      "'443.81%'", 
      "'443.9%'", 
      "'892.1%'"), 
    where_ICD10_like = c(
        "'E08.51%'", 
        "'E08.59%'", 
        "'E09.51%'", 
        "'E09.59%'",
        "'E08.621%'", "'E09.621%'", "'E10.621%'", "'E11.621%'", "'E13.621%'", 
        "'I72.4%'",
        "'I70.21%'",
        "'I73.89%'",
        "'I73.9%'",
        "'S91.3%'"),
    name = "dcsi.pvd1")

dcsi.pvd2 <- extract_condition(
    where_ICD9_like = c(
        "'040.0%'",
        "'444.22%'",  
        "'707.1%'", 
        "'785.4%'"),
    where_ICD10_like = c(
        "'A48.0%'", 
        "'I74.3%'",
        "'L97%'",
        "'E08.52%'",  "'E09.52%'", "'E10.52%'", "'E11.52%'", "'E13.52%'",
        "'I96%'"), 
    name = "dcsi.pvd2")

In [ ]:
dcsi.pvd1$first %>% count(dcsi.pvd1_concept_source_code, dcsi.pvd1_concept_name, sort=T)

In [ ]:
dcsi.pvd2$first %>% count(dcsi.pvd2_concept_source_code, dcsi.pvd2_concept_name, sort=T)

### Metabolic subcomponent (hypoglycemia, ketoacidosis, hyperosmolarity, coma)

In [ ]:
dcsi.met1 <- extract_condition(
    where_ICD9_like = c("'NOTAPPLICABLE%'"), 
    where_ICD10_like = c(
        "'E08.00%'", "'E08.10%'",
        "'E09.00%'", "'E09.10%'",
        "'E10.00%'", "'E10.10%'",
        "'E11.00%'", "'E11.10%'",
        "'E13.00%'", "'E13.10%'",
        "'E08.649%'", "'E09.649%'", "'E10.649%'", "'E11.649%'", "'E13.649%'"), 
    name = "dcsi.met1")

dcsi.met2 <- extract_condition(
    where_ICD9_like = c("'249.1%'", "'249.2%'", "'249.3%'", 
                        "'250.1%'", "'250.2%'", "'250.3%'"),
    where_ICD10_like = c(
        "'E08.01%'", "'E08.11%'",
        "'E09.01%'", "'E09.11%'",
        "'E10.01%'", "'E10.11%'",
        "'E11.01%'", "'E11.11%'",
        "'E13.01%'", "'E13.11%'",
        "'E08.641%'", "'E09.641%'", "'E10.641%'", "'E11.641%'", "'E13.641%'"), 
    name = "dcsi.met2")

In [ ]:
dcsi.met1$first %>% count(dcsi.met1_concept_source_code, dcsi.met1_concept_name)

In [ ]:
dcsi.met2$first %>% count(dcsi.met2_concept_source_code, dcsi.met2_concept_name)

## Merging time-dependent components

Create the tmerge object

In [ ]:
#Use tmerge to get the time-dependent dataset.  Use arbitrary start and endpoints, then convert back to date. 

#arbitrary origin
ogn <- as.Date("1800-01-30")
ogn

In [ ]:
ids_sql <- paste("
    SELECT distinct person.person_id
    FROM `person` person ", sep="")

ids_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  "person_ids",
  "ids.csv")

bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), ids_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  ids_path,
  destination_format = "CSV")

read_bq_export_from_workspace_bucket <- function(export_path) {
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), show_col_types = FALSE)
          chunk
        }))
}
ids_df <- read_bq_export_from_workspace_bucket(ids_path)

dim(ids_df)

In [ ]:
write_to_bucket(ids_df, "person_ids.csv")

In [ ]:
IDs <- ids_df

In [ ]:
#Initialize the tmerge object 

tmp <- IDs %>% mutate(time = 1000, status = 0) %>% arrange(person_id)

tindpt <- tmerge(tmp, tmp, id = person_id, endpt = event(time, status))

In [ ]:
tdept <- slip_condition_first_DCSI("dcsi.ret1")

In [ ]:
tdept <- slip_condition_DCSI("dcsi.ret2")
tdept <- slip_condition_DCSI("dcsi.neph1")
tdept <- slip_condition_DCSI("dcsi.neph2")
tdept <- slip_condition_DCSI("dcsi.neur1")
tdept <- slip_condition_DCSI("dcsi.cbr1")
tdept <- slip_condition_DCSI("dcsi.cbr2")
tdept <- slip_condition_DCSI("dcsi.cvd1")
tdept <- slip_condition_DCSI("dcsi.cvd2")
tdept <- slip_condition_DCSI("dcsi.pvd1")
tdept <- slip_condition_DCSI("dcsi.pvd2")
tdept <- slip_condition_DCSI("dcsi.met1")
tdept <- slip_condition_DCSI("dcsi.met2")

In [ ]:
saveRDS(tdept, "tdept_tmp.RDS")

## Clean and save the time-dependent DCSI score

In [ ]:
firstvisit_df <- read_from_bucket("first_condition_any.csv")

In [ ]:
tdept_cleaned <- tdept %>% 
    select(-c(time, status, tstop, endpt)) %>%
    full_join(firstvisit_df)
tdept_cleaned[is.na(tdept_cleaned)] <- 0

nrow(tdept_cleaned)
length(unique(tdept_cleaned$person_id))

Convert time back into date format:

In [ ]:
tdept_cleaned <- tdept_cleaned %>%
    # require small offset to not be 1s before correct date
    mutate(tstart = as.Date(date_decimal(decimal_date(ogn) + tstart + 5E-8)), 
           #below line only replaces initial T0 with true first observation time. Later rows not affected
           tstart = pmax(tstart, first_condition_date)) %>% #Can create duplicate tstarts though, with event on 2nd
    distinct()

In [ ]:
tdept_cleaned2 <- tdept_cleaned %>%
    #calculate DCSI score
    mutate(across(contains("2"), function(x){ x*2})) %>%
    mutate(ret = pmax(ret1, ret2), 
           neph = pmax(neph1, neph2),
           cbr = pmax(cbr1, cbr2),
           cvd = pmax(cvd1, cvd2),
           pvd = pmax(pvd1, pvd2),
           met = pmax(met1, met2),
           DCSIscore = neur1 + ret + neph + cbr + cvd + pvd + met)    

In [ ]:
DCSI <- tdept_cleaned2 %>%
    #if first condition (T0) was a DSCI condition, there will be 2 rows. Keep the one with the conditions
    #Remove the one with all 0s
    group_by(person_id, tstart) %>%
    slice_max(DCSIscore) %>%
    select(-first_condition_date, -contains(c("1", "2"))) %>%
    select(person_id, tstart, DCSIscore, everything()) %>%
    ungroup()

In [ ]:
write_to_bucket(DCSI, "DCSI_time_dependent.csv")

In [ ]:
suppressWarnings(
    rm(DCSI, dcsi.ret1, dcsi.ret2, dcsi.neph1, dcsi.neph2, dcsi.cbr1, dcsi.cbr2, dcsi.cvd1, dcsi.cvd2, 
       dcsi.pvd1, dcsi.pvd2, dcsi.met1, dcsi.met2, tdept_cleaned, tdept_cleaned2, firstvisit_df, ogn, tdept)
    )
gc()

# Medications

## BP-lowering drugs

### All BP-lowering drugs

First, extract all BP-lowering drugs. Then we will assign their class, including combination drugs belonging to multiple classes.

In [ ]:
med_concepts_key["BPDrug"]

In [ ]:
bp_lowering_df <- extract_medication(med_concepts_key["BPDrug"], "BPDrug")
dim(bp_lowering_df)

In [ ]:
bp_lowering_df %>% count(route_concept_name, sort=T) 

In [ ]:
bp_lowering_df <- bp_lowering_df %>%
    mutate(name_without_dose = gsub("[0-9]|MG|Oral|Tablet|Capsule|Extended|Release| |\\.|\\[.*\\]", "", 
                                    standard_concept_name)) %>%
    filter(is.na(route_concept_name) | 
           grepl("oral|intravenous|No matching concept|gastr|digest|enteral|arterial|intramuscular", 
                 route_concept_name, ignore.case=T))

bp_lowering_summary <- bp_lowering_df %>%
    count(name_without_dose, sort=T) %>% ungroup() 

In [ ]:
bp_lowering_check1 <- bp_lowering_summary %>% 
    filter(!grepl("olol|ilol|alol|artan|pril|diltiazem|ipine|semide|verapamil|hydrochlorothiazide", 
                  name_without_dose, ignore.case=T) &
            !grepl("spironolactone|hydralazine|chlorthalidone|zosin|bumetanide|clonidine|metolazone", 
                   name_without_dose, ignore.case=T) &
            !grepl("eplerenone|triamterene|chlorothiazide|methyldopa|pentoxifylline|indapamide", 
                   name_without_dose, ignore.case=T) &
            !grepl("amiloridehydrochloride|trichlormethiazide|amiloride|bethanidine|guanfacine", 
                   name_without_dose, ignore.case=T) &
            !grepl("aliskiren|ethacryn|sodiumnitroprusside|bendroflumethiazide|nitroprusside", 
                   name_without_dose, ignore.case=T) &
            !grepl("methyclothiazide|guanethidine", name_without_dose, ignore.case=T))

In [ ]:
opth <- bp_lowering_summary %>% 
    anti_join(bp_lowering_check1) %>%
    filter(grepl("OphthalmicSolution", name_without_dose))

In [ ]:
bp_lowering_EXCLUDE_names <- bp_lowering_check1 %>%
    rbind(opth) %>%
    select(name_without_dose)
nrow(bp_lowering_EXCLUDE_names)

In [ ]:
bp_lowering_EXCLUDE_concepts <- 
    inner_join(bp_lowering_EXCLUDE_names, 
              bp_lowering_df %>% distinct(drug_concept_id, standard_concept_name, name_without_dose))
nrow(bp_lowering_EXCLUDE_concepts)

In [ ]:
bp_lowering_df <- bp_lowering_df %>%
    anti_join(bp_lowering_EXCLUDE_concepts)

In [ ]:
write_to_bucket(bp_lowering_df, "bp_lowering_drugs.csv")

### Ace Inhibitors
We need to capture combination drugs, however, each incredient is a descendant concept, so many drugs are inapppropriately captured.  We simply grep "pril" from the concept name to ensure that there is truly an ACE Inhibitor as an ingredient. 

In [ ]:
ACEInhibitors_df <- bp_lowering_df %>%
    filter(grepl(med_patterns_key["ACEInhibitors"], standard_concept_name, ignore.case=T))

In [ ]:
ACEInhibitors_summary <- ACEInhibitors_df %>% 
    count(name_without_dose, sort=T) 

In [ ]:
ACEInhibitors_df %>% count(route_concept_name, sort=T)

In [ ]:
ACEInhibitors_df <- ACEInhibitors_df %>% 
    filter(route_concept_name %in% c("Oral route", NA, "No matching concept", "Intravenous route"))

In [ ]:
ACEInhibitors_df %>% count(name_without_dose, sort=T)

In [ ]:
# check <- ACEInhibitors_df %>% count(name_without_dose, sort=T) %>% 
#     filter(!(
#         grepl("lisinopril|trandolapril|ramipril|enalapril|perindopril|moexipril|captopril|fosinopril|quinapril|benazepril", 
#               name_without_dose)))
# check

In [ ]:
write_to_bucket(ACEInhibitors_df, "ACEIhhibitors.csv")

### Angiogensin II Receptor Blockers
We will grep 'artan' to filter out the inappropriate descendant concepts/ingrediants. 

In [ ]:
AngiotensinIIRB_df <- bp_lowering_df %>%
    filter(grepl(med_patterns_key["AngiotensinIIRB"], standard_concept_name, ignore.case=T)) 

In [ ]:
AngiotensinIIRB_summary <- AngiotensinIIRB_df %>% count(name_without_dose, sort=T)

In [ ]:
AngiotensinIIRB_df <- AngiotensinIIRB_df %>% 
    filter(route_concept_name %in% c("Oral route", NA, "No matching concept", "Intravenous route")) 

In [ ]:
AngiotensinIIRB_df %>% count(name_without_dose, sort=T)

In [ ]:
write_to_bucket(AngiotensinIIRB_df, "AngiotensinIIRB.csv")

### Calcium Channel Blockers

In [ ]:
CChannelBlockers_df <- bp_lowering_df %>% 
    filter(grepl(med_patterns_key["CChannelBlockers"], standard_concept_name, ignore.case=T)) %>% 
    filter(route_concept_name %in% c("Oral route", NA, "No matching concept", "Intravenous route")) 

In [ ]:
CChannelBlockers_summary <- CChannelBlockers_df %>% count(name_without_dose, sort=T)

In [ ]:
write_to_bucket(CChannelBlockers_df, "CalciumChannelBlockers.csv")

### Beta Blockers

In [ ]:
BetaBlockers_df <- bp_lowering_df %>%
    filter(grepl(med_patterns_key["BetaBlockers"], standard_concept_name, ignore.case=T))

In [ ]:
BetaBlockers_summary <- BetaBlockers_df %>%
    count(name_without_dose, sort=T)

In [ ]:
write_to_bucket(BetaBlockers_df, "BetaBlockers.csv")

### Diuretics

In [ ]:
Diuretics_df <- bp_lowering_df %>%
    filter(grepl(med_patterns_key["Diuretics"], standard_concept_name, ignore.case = T) )

In [ ]:
Diuretics_summary <- Diuretics_df %>%
    count(name_without_dose, sort=T)
Diuretics_summary

In [ ]:
write_to_bucket(Diuretics_df, "Diuretics.csv")

### Other bp-lowering drugs (vasodilators, etc)

In [ ]:
uncategorized_bp_lowering <- bp_lowering_df %>%
    anti_join(
        rbind(ACEInhibitors_df %>% distinct(drug_concept_id),
             AngiotensinIIRB_df %>% distinct(drug_concept_id),
             CChannelBlockers_df %>% distinct(drug_concept_id),
             BetaBlockers_df %>% distinct(drug_concept_id),
             Diuretics_df %>% distinct(drug_concept_id))
    )

In [ ]:
uncategorized_bp_lowering_summary <- uncategorized_bp_lowering %>%
    count(name_without_dose, sort=T)
uncategorized_bp_lowering_summary

In [ ]:
rm(ACEInhibitors_df, ACEInhibitors_summary, AngiotensinIIRB_df, AngiotensinIIRB_summary,
    BetaBlockers_df, BetaBlockers_summary, bp_lowering_check1, bp_lowering_check2, bp_lowering_df,
    bp_lowering_EXCLUDE, bp_lowering_EXCLUDE_concepts, bp_lowering_EXCLUDE_names, bp_lowering_path,
    bp_lowering_sql, bp_lowering_summary, CChannelBlockers_df, CChannelBlockers_summary, check, chk,
    Diuretics_df, Diuretics_summary, opth, tmp_path, uncategorized_bp_lowering,
    uncategorized_bp_lowering_summary
)

## Diabetes Drugs

### Insulin

In [ ]:
med_concepts_key["Insulin"]

In [ ]:
insulin_df <- extract_medication(med_concepts_key["Insulin"], "Insulin")

In [ ]:
insulin_df %>% count(route_concept_name, sort=T)

In [ ]:
insulin_df <- insulin_df %>% 
    mutate(name_without_dose = gsub("[0-9]|\\/ML|ML|Pen Injector|Injectable Solution|MG|Oral|Tablet|Capsule|Extended|Release| |\\.|\\[.*\\]|UNT\\/ML", "", 
                                    standard_concept_name, ignore.case = TRUE)) %>%
    filter(is.na(route_concept_name) | 
           grepl("cutaneous|No matching concept|Intravenous|Intramuscular|Intraduodenal|Route of administration|oral|inhalation", 
                 route_concept_name, ignore.case=T))

In [ ]:
insulin_summary <- insulin_df %>%
    count(name_without_dose, sort=T) %>% ungroup() 
insulin_summary

In [ ]:
not_insulin <- insulin_df %>%
    filter(name_without_dose %in% c("liraglutide", "lixisenatide", "{(lixisenatide)/(lixisenatide)}Pack")) %>%
    count(name_without_dose, drug_concept_id, standard_concept_name, sort=T)
not_insulin

In [ ]:
insulin_df <- insulin_df %>%
    anti_join(not_insulin)

In [ ]:
write_to_bucket(insulin_df, "insulin.csv")

### Metformin

In [ ]:
med_concepts_key["Metformin"]

In [ ]:
metformin_df <- extract_medication(med_concepts_key["Metformin"], "Metformin")

In [ ]:
metformin_df <- metformin_df %>% 
    mutate(name_without_dose = gsub("Suspension|Solution|24 HR|\\/ML|[0-9]|MG|Oral|Tablet|Capsule|Extended|Release| |\\.|\\[.*\\]", "", 
                                    standard_concept_name), ignore.case=T) %>%
    filter(is.na(route_concept_name) | 
               route_concept_name %in% c("Oral route", "No matching concept", "Route of administration not applicable"))

In [ ]:
metformin_summary <- metformin_df %>%
    count(name_without_dose, sort=T) %>% ungroup() 

In [ ]:
metformin_summary %>% 
    mutate(metformin_only = as.numeric(!grepl("liptin|glyburide|glitazone|flozin|izide|glinide", name_without_dose))) %>%
    arrange(metformin_only, desc(n))

In [ ]:
metformin_df <- metformin_df %>% 
    mutate(ComboDrug = grepl("liptin|glyburide|glitazone|flozin|izide|glinide", name_without_dose))

In [ ]:
write_to_bucket(metformin_df, "metformin.csv")

### All oral glucose-lowering drugs except insulin, metformin

#### All together

In [ ]:
med_concepts_key["DMDrug"]

In [ ]:
oraldm_df <- extract_medication(med_concepts_key["DMDrug"], "DMDrug")

In [ ]:
oraldm_df <- oraldm_df %>% 
    mutate(name_without_dose = gsub("Pen Injector|Injectable Solution|Suspension|Solution|24 HR|\\/ML|ML|[0-9]|MG|Oral|Tablet|Capsule|Extended|Release| |\\.|\\[.*\\]", "", 
                                    standard_concept_name)) %>% 
    filter(is.na(route_concept_name) | 
               route_concept_name %in% c("Oral route", "No matching concept", "Route of administration not applicable")) %>%
    filter(!(grepl("Injection", standard_concept_name, ignore.case=T)))

In [ ]:
oraldm_summary <- oraldm_df %>%
    count(name_without_dose, sort=T) %>% ungroup() 

In [ ]:
#Many statins made their way in due to being an ingredient in combo drugs
#Same with metformin

In [ ]:
oraldm_summary_excl <- oraldm_summary %>% 
    filter(grepl("statin", name_without_dose) | name_without_dose %in% 
           c("metforminhydrochloride", "metformin", "Osmoticmetforminhydrochloride", "Modifiedmetforminhydrochloride",
            "liraglutide", "dulaglutide", "pramlintide", "pramlintideacetate", "albiglutide")) %>% #injectables
    filter(!(grepl("gliptin", name_without_dose))) #keep gliptins
oraldm_summary_excl

In [ ]:
oraldm_summary %>% anti_join(oraldm_summary_excl) %>% 
    #semaglutide is a GLP1 that can be taken orally or injected
    filter(!(grepl("liptin|glitazone|glyburide|glipizide|glimepiride|flozin|glinide|semaglutide|exenatide", 
                   name_without_dose)))

In [ ]:
oraldm_df <- oraldm_df %>%
    anti_join(oraldm_summary_excl)

In [ ]:
oraldm_df %>% count(name_without_dose, sort=T)

In [ ]:
write_to_bucket(oraldm_df, "oral_dmdrug_excl_insulin_metformin.csv")

#### Separate out by class

##### Sulfonylurea

In [ ]:
med_patterns_key["Sulfonylurea"]

In [ ]:
Sulfonylurea <- oraldm_df %>% 
    filter(grepl(med_patterns_key["Sulfonylurea"], standard_concept_name, ignore.case=T))

In [ ]:
Sulfonylurea %>% count(name_without_dose, sort=T)

##### DPP4

In [ ]:
med_patterns_key["DPP4"]

In [ ]:
DPP4 <- oraldm_df %>% 
    filter(grepl(med_patterns_key["DPP4"], standard_concept_name, ignore.case=T))

In [ ]:
DPP4 %>% count(name_without_dose, sort=T)

##### SGLT2

In [ ]:
med_patterns_key["SGLT2"]

In [ ]:
SGLT2 <- oraldm_df %>% 
    filter(grepl(med_patterns_key["SGLT2"], standard_concept_name, ignore.case=T))

In [ ]:
SGLT2 %>% count(name_without_dose, sort=T)

##### Thiazoladinedione

In [ ]:
med_patterns_key["Thiazoladinedione"]

In [ ]:
Thiazoladinedione <- oraldm_df %>% 
    filter(grepl(med_patterns_key["Thiazoladinedione"], standard_concept_name, ignore.case=T))

In [ ]:
Thiazoladinedione %>% count(name_without_dose, sort=T)

##### GLP1 ORAL ONLY

Usually these are injectables. Route could be incorrectly marked

In [ ]:
med_patterns_key["GLP1"]

In [ ]:
GLP1_ORAL <- oraldm_df %>% 
    filter(grepl(med_patterns_key["GLP1"], standard_concept_name, ignore.case=T))

In [ ]:
GLP1_ORAL %>% count(name_without_dose, sort=T)

In [ ]:
GLP1_ORAL %>% count(route_concept_name, sort=T)

In [ ]:
GLP1_ORAL <- GLP1_ORAL %>% 
    filter(route_concept_name == "Oral route") %>%
    filter(standard_concept_name == "semaglutide")

##### Other Oral glucose lowering drugs

In [ ]:
Other_oraldm <- oraldm_df %>%
    anti_join(Sulfonylurea) %>%
    anti_join(DPP4) %>%
    anti_join(SGLT2) %>%
    anti_join(Thiazoladinedione) %>%
    anti_join(GLP1_ORAL) %>%
    filter(name_without_dose != "exenatide")

In [ ]:
Other_oraldm %>% count(name_without_dose, route_concept_name, sort=T)

##### Write out subclasses of oral DM meds

In [ ]:
write_to_bucket(Sulfonylurea, "Sulfonylurea.csv")
write_to_bucket(DPP4, "DPP4.csv")
write_to_bucket(SGLT2, "SGLT2.csv")
write_to_bucket(Thiazoladinedione, "Thiazoladinedione.csv")
write_to_bucket(GLP1_ORAL, "GLP1_ORAL.csv")
write_to_bucket(Other_oraldm, "Other_oraldm.csv")

##### Add injected GLP1

In [ ]:
oraldm_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  #strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "DMDrug",
  "DMDrug_*.csv")

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), drug_type_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
oraldm_df_all <- read_bq_export_from_workspace_bucket(oraldm_path)

In [ ]:
GLP1_INJ <- oraldm_df_all %>%
    filter(grepl(med_patterns_key["GLP1"], standard_concept_name, ignore.case=T)) %>% 
    mutate(name_without_dose = gsub("Pen Injector|Injectable Solution|Injection|Suspension|Solution|24 HR|\\/ML|ML|[0-9]|MG|Oral|Tablet|Capsule|Extended|Release| |\\.|\\[.*\\]", "", 
                                    standard_concept_name)) %>%
    filter(is.na(route_concept_name) | 
          route_concept_name %in% c("No matching concept", "Route of administration not applicable") | 
          route_concept_name %in% c("Subcutaneous route	", "Intravenous route", "Intramuscular route", "Intraepidermal route")  
          ) 

In [ ]:
GLP1_INJ %>% count(name_without_dose, route_concept_name, sort=T)

In [ ]:
GLP1_INJ %>% count(standard_concept_name, sort=T)

In [ ]:
write_to_bucket(GLP1_INJ, "GLP1_INJECTED.csv")

In [ ]:
rm(Sulfonylurea, DPP4, SGLT2, Thiazoladinedione, GLP1_ORAL, Other_oraldm, GLP1_INJ, oraldm_df, oraldm_df_all)
gc()

### Glucagon

In [ ]:
med_concepts_key["Glucagon"]

In [ ]:
glucagon_df <- extract_medication(med_concepts_key["Glucagon"], "Glucagon")

In [ ]:
glucagon_df %>% count(route_concept_name, sort=T)

In [ ]:
glucagon_df <- glucagon_df %>% 
    mutate(name_without_dose = gsub("[0-9]|MG|Oral|Tablet|Capsule|Extended|Release| |\\.|\\[.*\\]", "", 
                                    standard_concept_name)) %>%
    filter(is.na(route_concept_name) | 
          route_concept_name %in% c("No matching concept", "Intravenous route", "Intramuscular route",
                                    "Subcutaneous route", "Nasal route", "Route of administration value"))

In [ ]:
glucagon_summary <- glucagon_df %>%
    count(name_without_dose, sort=T) %>% ungroup() 
glucagon_summary

In [ ]:
write_to_bucket(glucagon_df, "glucagon.csv")

In [ ]:
rm(glucagon_df,  glucagon_summary, insulin_df,  
   insulin_summary, metformin_df, metformin_summary, not_insulin, oraldm_df, 
   oraldm_summary, oraldm_summary_excl)
gc()

## Lipids-modifying agents

### Extract all together

In [ ]:
med_concepts_key["LipidDrug"]

In [ ]:
lipids_mod_agents_df <- extract_medication(med_concepts_key["LipidDrug"], "LipidDrug")

In [ ]:
lipids_mod_agents <- lipids_mod_agents_df %>% 
    distinct(standard_concept_name) %>%
    mutate(name_without_dose = gsub("12 HR|24 HR|[0-9]|MG|Oral|Tablet|Capsule|Delayed|Extended|Release|Effervescent|Solution|PowderforSuspension|Suspension|Powder|Disintegrating| |\\.|\\[.*\\]", "", 
                                    standard_concept_name, ignore.case=T))  %>%
    right_join(lipids_mod_agents_df %>% select(-contains("name_without_dose")) %>% distinct())

lipids_mod_agents_summary <- lipids_mod_agents %>% 
    count(name_without_dose, sort=T) %>% ungroup() 
lipids_mod_agents_summary

A lot of the wrong drugs ended up there because of ingrediants in combo drugs. We'll clean this below

### Statins (and dose classes)

In [ ]:
med_patterns_key["Statin_Any"]

In [ ]:
statins_summary <- lipids_mod_agents_summary %>% 
    filter(grepl(med_patterns_key["Statin_Any"], name_without_dose, ignore.case=T)) 
statins_summary

In [ ]:
statins <- statins_summary %>% select(-n) %>% 
    inner_join(lipids_mod_agents)

Add statins doses

In [ ]:
dose_patterns_key["Statin_High"]
dose_patterns_key["Statin_Medium"]
dose_patterns_key["Statin_Low"]

In [ ]:
statins <- statins %>% 
    distinct(standard_concept_name) %>%
    mutate(statin_dose_class = case_when(
        grepl(dose_patterns_key["Statin_High"], standard_concept_name, ignore.case=T) ~ "High",
        grepl(dose_patterns_key["Statin_Medium"], standard_concept_name, ignore.case=T) ~ "Medium",
        grepl(dose_patterns_key["Statin_Low"], standard_concept_name, ignore.case=T) ~ "Low",
        TRUE ~ NA)) %>%
    right_join(statins %>% select(-contains("statin_dose_class")))

statins %>% group_by(statin_dose_class) %>% summarize(prop = n()/nrow(statins))

In [ ]:
statins %>% filter(is.na(statin_dose_class)) %>% count(standard_concept_name, sort=T)

In [ ]:
statins <- statins %>%
    mutate(statin_dose_class = coalesce(statin_dose_class, "Unknown"))

In [ ]:
#write_to_bucket(statins, "Statins_DoseClass.csv")

### Niacin

In [ ]:
med_patterns_key["Niacin"]

In [ ]:
niacin_summary <- lipids_mod_agents_summary %>% 
    filter(grepl(med_patterns_key["Niacin"], name_without_dose, ignore.case=T))
niacin_summary
#many of these are probably vitamin supplements

In [ ]:
niacin_keep_summary <- niacin_summary %>% 
    filter(grepl("statin", name_without_dose) | nchar(name_without_dose) < 50)
niacin_keep_summary

In [ ]:
niacin <- niacin_keep_summary %>%
    select(-n) %>%
    inner_join(lipids_mod_agents)

### Fatty acids

In [ ]:
med_patterns_key["FattyAcid"]

In [ ]:
fatty_acids_summary <- lipids_mod_agents_summary %>%
    filter(grepl(med_patterns_key["FattyAcid"], name_without_dose, ignore.case=T))
fatty_acids_summary

In [ ]:
fatty_acids <- fatty_acids_summary %>%
    select(-n) %>%
    inner_join(lipids_mod_agents)

### Ezetimibe

In [ ]:
med_patterns_key["Ezetimibe"]

In [ ]:
Ezetimibe_summary <- lipids_mod_agents_summary %>%
    filter(grepl(med_patterns_key["Ezetimibe"], name_without_dose, ignore.case=T))
Ezetimibe_summary

In [ ]:
Ezetimibe <- Ezetimibe_summary %>%
    select(-n) %>%
    inner_join(lipids_mod_agents)

### BAS (Bile acid sequestrants)

In [ ]:
med_patterns_key["BAS"]

In [ ]:
BAS_summary <- lipids_mod_agents_summary %>%
    filter(grepl(med_patterns_key["BAS"], name_without_dose, ignore.case=T))
BAS_summary

In [ ]:
BAS <- BAS_summary %>%
    select(-n) %>%
    inner_join(lipids_mod_agents)

### Fibrates

In [ ]:
med_patterns_key["Fibrates"]

In [ ]:
Fibrates_summary <- lipids_mod_agents_summary %>%
    filter(grepl(med_patterns_key["Fibrates"], name_without_dose, ignore.case=T))
Fibrates_summary

In [ ]:
Fibrates <- Fibrates_summary %>%
    select(-n) %>%
    inner_join(lipids_mod_agents)

### Other lipid lowering

In [ ]:
other_lipid_lowering <- lipids_mod_agents %>% 
    anti_join(statins) %>% 
    anti_join(niacin_summary) %>% 
    #anti_join(lipids_ingredients_excl) %>%
    anti_join(fatty_acids) %>%
    anti_join(BAS) %>%
    anti_join(Ezetimibe) %>%
    anti_join(Fibrates) 

other_lipid_lowering_summary <- other_lipid_lowering %>%
    count(name_without_dose, sort=T) 

In [ ]:
lipids_ingredients_excl <- other_lipid_lowering_summary %>% 
    filter(grepl("lisinopril|amlodipine|aspirin|artan|pril|indapamide", name_without_dose)) 

In [ ]:
lipids_ingredients_excl

In [ ]:
other_lipid_lowering <- other_lipid_lowering %>% 
    anti_join(lipids_ingredients_excl)

other_lipid_lowering_summary <- other_lipid_lowering %>%
    count(name_without_dose, sort=T)
other_lipid_lowering_summary

In [ ]:
other_lipid_lowering <- other_lipid_lowering %>% 
    filter(!name_without_dose %in% c("pyridoxal", "Pyridoxal", 
                                     "aceticacid/ML/antipyrine/ML/benzocaine/ML/policosanol/MLOticSolution"))

### Combine and write out

In [ ]:
statins_key <- statins %>% distinct(standard_concept_name, name_without_dose, statin_dose_class) %>% mutate(Statin=T)

lipids_lowering_key <- niacin %>% distinct(name_without_dose) %>% mutate(Niacin=T) %>%
                      full_join(fatty_acids %>% distinct(name_without_dose) %>% mutate(FattyAcid=T)) %>%
                      full_join(BAS %>% distinct(name_without_dose) %>% mutate(BAS=T)) %>%
                      full_join(Ezetimibe %>% distinct(name_without_dose) %>% mutate(Ezetimibe=T)) %>%
                      full_join(Fibrates %>% distinct(name_without_dose) %>% mutate(Fibrates=T)) %>%
                      full_join(other_lipid_lowering %>% distinct(name_without_dose) %>% mutate(OtherLipidLowering=T)
                    ) 
lipids_lowering_key

In [ ]:
lipids_lowering <- lipids_mod_agents %>%
    inner_join(lipids_lowering_key) %>%
    full_join(
        lipids_mod_agents %>% inner_join(statins_key)
    )

In [ ]:
lipids_lowering %>% count(Statin, Niacin, FattyAcid, BAS, Ezetimibe, Fibrates, OtherLipidLowering, sort=T)

In [ ]:
included_concepts <- lipids_lowering %>% count(name_without_dose)
included_concepts

In [ ]:
excluded_concepts <- lipids_mod_agents_summary %>%
    anti_join(included_concepts %>% select(-n))

In [ ]:
excluded_concepts

In [ ]:
write_to_bucket(lipids_lowering, "lipids_lowering_agents_v2.csv")

## Anticoagulants

In [ ]:
med_concepts_key["Anticoagulants"]

In [ ]:
Anticoagulants_df <- extract_medication(med_concepts_key["Anticoagulants"], "Anticoagulants")

In [ ]:
Anticoagulants_df <- Anticoagulants_df %>%
    mutate(name_without_dose = gsub("\\}|\\)|\\{|\\(|,|/|sodium|porcine|bovine|pack|Cartridge|Injection|Prefilled Syringe|UNT/ML|ML|Injectable Solution|[0-9]|MG|Oral|Tablet|Capsule|Extended|Release| |\\.|\\[.*\\]", "", 
                                    tolower(standard_concept_name),ignore.case=T)) 

In [ ]:
Anticoagulants_summary <- Anticoagulants_df %>%
    count(name_without_dose, sort=T) %>% ungroup() 

In [ ]:
Anticoagulants_summary 

In [ ]:
write_to_bucket(Anticoagulants_df, "Anticoagulants.csv")

## Aspirin

In [ ]:
med_concepts_key["Aspirin"]

In [ ]:
Aspirin_df <- extract_medication(med_concepts_key["Aspirin"], "Aspirin")

In [ ]:
Aspirin_df %>% count(standard_concept_name, sort=T) 

In [ ]:
Aspirin_df %>% count(standard_concept_name, sort=T) %>%
    filter(!grepl("aspirin", standard_concept_name, ignore.case=T))

In [ ]:
write_to_bucket(Aspirin_df, "Aspirin.csv")

## Corticosteroids

In [ ]:
med_concepts_key["Corticosteroids"]

In [ ]:
Corticosteroids_df <- extract_medication(med_concepts_key["Corticosteroids"], "Corticosteroids")

We only want Oral/Injectable corticosteroids;, not topical, inhaled, or ophthalmic

In [ ]:
Corticosteroids_df <- Corticosteroids_df %>% 
    mutate(name_without_dose = gsub("\\[.*?\\]", "", standard_concept_name)) %>%
    mutate(name_without_dose = gsub("Pen Injector|Injectable Solution|Injectable|Drug|Implant|Prefilled|Syringe|Injection|Suspension|delayed|pack|granules|Solution|24 HR|\\/ML|ML|[0-9]|MG|Oral|Tablet|Capsule|phosphate|diacetate|acetate|hexacetonide|acetonide|sodium|butyrate|valerate|furoate|disintegrating|Effervescent|Extended|Release| |\\.|\\{|\\}|\\(|\\)", "", 
                                    name_without_dose, ignore.case=T)) %>%
    mutate(name_without_dose = gsub("\\/$", "", name_without_dose)) %>%
    mutate(name_without_dose = gsub("\\/$", "", name_without_dose))

In [ ]:
Corticosteroids_df %>% count(route_concept_name, sort=T)

In [ ]:
Corticosteroids_df <- Corticosteroids_df %>%
    filter(route_concept_name %in% c('Oral route', 
                                     'No matching concept', 'Intravenous route', NA, 
                                     'Intramuscular route', 'Subcutaneous route', 
                                     'Intrabursal route', 
                                     'Nasogastric route', 
                                    'Gastrostomy route', 
                                     'Intravascular route'))

In [ ]:
Corticosteroids_df <- Corticosteroids_df %>% 
    filter(!grepl("topical|cream|nasal|spray|Ophth|otic|inhal|ACTUAT|paste", standard_concept_name, ignore.case=T)) %>%
    mutate(name_without_dose = tolower(name_without_dose))

In [ ]:
Corticosteroids_df %>% count(name_without_dose, sort=T)

In [ ]:
Corticosteroids_df <- Corticosteroids_df %>%
    mutate(name_without_dose = gsub("betamethasone/betamethasone", "betamethasone", name_without_dose)) 

In [ ]:
write_to_bucket(Corticosteroids_df, "Corticosteroids_systemic.csv")

## Proton Pump Inhibitors

In [ ]:
med_concepts_key["ProtonPumpInhibitors"]

In [ ]:
ProtonPumpInhibitors_df <- extract_medication(med_concepts_key["ProtonPumpInhibitors"], "ProtonPumpInhibitors")

In [ ]:
 ProtonPumpInhibitors_df %>% count(route_concept_name, sort=T)

In [ ]:
ProtonPumpInhibitors <- ProtonPumpInhibitors_df %>% 
    filter(route_concept_name %in% c(
                "Oral route",
                "Intravenous route",
                NA,
                "No matching concept",
                "Intraepidermal route", 
                "Route of administration not applicable",
                "Nasogastric route", 
                "Gastrostomy route", 
                "Gastro-intestinal stoma route", 
                "Enteral route", 
                "Gastroenteral route", 
                "Digestive tract route", 
                "PEG tube route",
                "Route of administration value", 
                "Orogastric route",
                "Jejunostomy route")
           )

In [ ]:
ProtonPumpInhibitors <- ProtonPumpInhibitors %>% 
    mutate(name_without_dose = gsub("\\[.*?\\]", "", standard_concept_name)) %>%
    mutate(name_without_dose = gsub("Pen Injector|Injectable Solution|Suspension|powder|Solution|Delayed|Disintegrating|injection|sodium|bicarbonate|strontium| for |granules|pack|24 HR|\\/ML|ML|[0-9]|MG|Oral|Tablet|Capsule|Extended|Release| |\\.|\\(|\\)|\\{|\\}", "", 
                                    name_without_dose, ignore.case=T)) %>%
    mutate(name_without_dose = gsub("\\/$", "", name_without_dose)) %>%
    mutate(name_without_dose = tolower(name_without_dose))

In [ ]:
 ProtonPumpInhibitors %>% count(name_without_dose, sort=T)

In [ ]:
write_to_bucket(ProtonPumpInhibitors, "ProtonPumpInhibitors.csv")

## Antipsychotics

In [ ]:
med_concepts_key["Antipsychotics"]

In [ ]:
Antipsychotics_df <- extract_medication(med_concepts_key["Antipsychotics"], "Antipsychotics")

In [ ]:
Antipsychotics <- Antipsychotics_df %>% 
    mutate(name_without_dose = gsub("\\[.*?\\]", "", standard_concept_name)) %>%
    mutate(name_without_dose = gsub("Sensor |Pen Injector|Injectable Solution|Suspension|powder|Solution|rectal|suppository|Disintegrating|injection|Delayed|granules|pack|sublingual|decanoate|enanthate|palmitate|24 HR|\\/ML|ML|[0-9]|MG|Oral|Tablet|Capsule|Extended|Release|prefilled|syringe|hydrochloride| |\\.|\\(|\\)|\\{|\\}", "", 
                                    name_without_dose, ignore.case=T)) %>% 
    mutate(name_without_dose = gsub("\\/$", "", name_without_dose)) %>%
    mutate(name_without_dose = tolower(name_without_dose))

In [ ]:
Antipsychotics <- Antipsychotics %>%
    mutate(name_without_dose = gsub("quetiapine/quetiapine/quetiapine", "quetiapine", name_without_dose)) %>%
    mutate(name_without_dose = gsub("cariprazine/cariprazine", "cariprazine", name_without_dose)) %>%
    mutate(name_without_dose = gsub("iloperidone/iloperidone/iloperidone/iloperidone", "iloperidone", name_without_dose)) 

In [ ]:
Antipsychotics %>% count(name_without_dose, sort=T) 

In [ ]:
Antipsychotics %>% count(route_concept_name, sort=T)

In [ ]:
Antipsychotics <- Antipsychotics %>%
    filter(route_concept_name %in% c(
        "Oral route",
        "Intravenous route",
        "No matching concept",
        NA, 
        "Intramuscular route",
        "Rectal route",
        "Sublingual route",
        "Route of administration not applicable",
        "Route of administration value",
        "Nasogastric route",
        "Gastrostomy route",
        "Enteral route",
        "Intravenous peripheral route"   
    )
)

In [ ]:
Antipsychotics %>% count(name_without_dose, sort=T)

In [ ]:
write_to_bucket(Antipsychotics, "Antipsychotics.csv")

## OpioidForPain_Rx

In [ ]:
med_concepts_key["OpioidForPain_Rx"]

In [ ]:
OpioidForPain_Rx_df <- extract_medication(med_concepts_key["OpioidForPain_Rx"], "OpioidForPain_Rx")

In [ ]:
OpioidForPain_Rx <- OpioidForPain_Rx_df %>% distinct(standard_concept_name) %>%
    mutate(name_without_dose = gsub("\\[.*?\\]", "", standard_concept_name)) %>%
    #remove misc info, also remove non-opioids that we don't care about
    mutate(name_without_dose = gsub("ibuprofen|acetaminophen|ropivacaine|chlorpheniramine|pheniramine|carisoprodol|bitartrate|diphenhydramine|pyrilamine|brompheniramine|propyphenazone|polistirex|guaifenesin|bupivacaine|droperidol|phenylephrine|promethazine|disint|buccal|mucosal|extract| usp |pseudoephedrine|bupropion|caffeine|triprolidine|aspirin|bella|donna|alkaloids|butalbital|matrix|delivery|Sensor |Pen Injector|Injectable Solution|Suspension|powder|Solution|rectal|suppository|terephthalate|egrating|injection|Delayed|granules|pack|sublingual|decanoate|enanthate|palmitate|phosphate|maleate|sulfate|sodium|salicylate|napsylate|tincture|film|transdermal|system|cartridge|nasal|spray|24 HR|24hr|12 HR|12hr|72 HR|72hr|mg/hr|\\/ML|ML|[0-9]|MG|Oral|Tablet|Capsule|Extended|Release|prefilled|metered|dose|actuat|syringe|lozenge|tartrate| for |abuse|deterrent|hydrochloride|-|,| |\\.|\\(|\\)|\\{|\\}", "", 
                                    name_without_dose, ignore.case=T)) %>% 
    mutate(name_without_dose = gsub("\\/$|^\\/", "", name_without_dose)) %>%
    mutate(name_without_dose = gsub("\\/$|^\\/", "", name_without_dose)) %>%
    mutate(name_without_dose = gsub("\\/$|^\\/", "", name_without_dose)) %>%
    mutate(name_without_dose = tolower(name_without_dose)) %>%
    filter(!grepl("Doverin|Nalox|ipecac|bupre", name_without_dose, ignore.case=T)) %>%
    mutate(name_without_dose = gsub("oxycodone/oxycodone", "oxycodone", name_without_dose)) %>% 
    inner_join(OpioidForPain_Rx_df)

nrow(OpioidForPain_Rx)

In [ ]:
OpioidForPain_Rx %>% count(name_without_dose, sort=T)

In [ ]:
write_to_bucket(OpioidForPain_Rx, "OpioidForPain_Rx.csv")

In [ ]:
rm(OpioidsForPain_Rx_df, OpioidForPain_Rx)
gc()

# Clean up

In [ ]:
toc()

In [ ]:
rm(list = ls())
gc()